In [ ]:
import torch
import numpy as np
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn.functional as F

class Autoencoder_merfish(nn.Module):
    def __init__(self, in_dim=34, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), in_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)
class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), in_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)


def recolor_image_by_type_centroid(rgb_img, type_map, save_path="./"):
    rgb_img = rgb_img.clone()
    type_map_np = type_map.cpu().numpy()
    img_np = (rgb_img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

    cell_types = sorted(set(np.unique(type_map_np)) - {0})
    recolored_np = np.ones_like(img_np) * 255

    for t in cell_types:
        mask = (type_map_np == t)
        pixels = img_np[mask]
        if len(pixels) == 0:
            continue
        mean_color = pixels.mean(axis=0).astype(np.uint8)
        recolored_np[mask] = mean_color

    recolored_img = torch.from_numpy(recolored_np).permute(2, 0, 1).float() / 255.0

    if save_path is not None:
        img_to_save = (recolored_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        Image.fromarray(img_to_save).save(save_path)
        print(f"Saved recolored image to {save_path}")

    return recolored_img


def print_cell_type_centroid_colors(img, type_map, img_name="Map"):
    if hasattr(img, 'cpu'):
        img = img.cpu().numpy()
    if hasattr(type_map, 'cpu'):
        type_map = type_map.cpu().numpy()
    if img.shape[0] == 3:
        img = np.transpose(img, (1, 2, 0))
    if img.max() <= 1.1:
        img = (img * 255).round().astype(np.uint8)

    H, W, _ = img.shape
    cell_types = sorted(set(np.unique(type_map)) - {0})

    print(f"--- {img_name} ---")
    fig, axes = plt.subplots(len(cell_types), 1, figsize=(6, 1.2 * len(cell_types)))
    if len(cell_types) == 1:
        axes = [axes]

    for i, t in enumerate(cell_types):
        mask = (type_map == t)
        mask = np.squeeze(mask)
        if not np.any(mask):
            continue
        colors = img[mask]  # [N, 3]
        center = colors.mean(axis=0)
        print(f"Cell type {t}: mean RGB={center.round(1).tolist()} (N={len(colors)})")
        # --- 可视化 ---
        ax = axes[i]
        rgb = center / 255  # matplotlib expects 0-1
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, color=rgb))
        txt = f"RGB={center.round(1).astype(int).tolist()}\nN={len(colors)}"
        ax.text(0.5, 0.5, txt, ha='center', va='center', fontsize=11,
                color='k' if np.mean(rgb) > 0.7 else 'w')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        ax.set_title(f"Cell type {t}", fontsize=12)
    plt.suptitle(f"{img_name} cell type mean colors")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

def infer_cell_map(latent_image, model):
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device)
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals
    #min_vals = torch.tensor([-62.822937, -50.404293, -74.085304], device=latent_image.device)
    #range_vals = torch.tensor([43.030205, 50.127666, 76.73948], device=latent_image.device) - min_vals
    H, W = latent_image.shape[1], latent_image.shape[2]
    #print(H,W)
    latent_image = latent_image.to("cuda")
    #latent_image = transfer(latent_image)
    #region_np = latent_image.permute(1, 2, 0).cpu().numpy()  # [512, 512]
    #plt.figure(figsize=(6, 6))
    #plt.imshow(region_np, cmap='tab20')  # 可选cmap, 分类色彩
    #plt.colorbar(label='Cell Type')
    #plt.axis('off')
    #plt.tight_layout()
    #plt.show()

    range_vals = range_vals.to("cuda")
    min_vals = min_vals.to("cuda")

    flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat_img == 1.0).all(dim=1)
    infer_input_rgb = flat_img[~white_mask]
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = model.decoder(z_recovered)
        #logits = F.log_softmax(logits,dim=1)
        #print(logits)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1

    pred = pred.reshape(1, H, W)
    return pred

def load_and_recover_z3d_png(path):
    """
    Load a PNG image as RGB, interpret it as normalized z_3d embedding encoded as uint8 [0–255],
    and recover the original z_3d float values via inverse scaling.
    """
    # 1. Load RGB image as uint8 tensor
    img = Image.open(path).convert("RGB")
    arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
    rgb = arr.view(img.size[1], img.size[0], 3).permute(2, 0, 1).clone()  # [3, H, W], uint8
    mask = (rgb > 235).all(dim=0)
    rgb[:, mask] = 255
    rgb_float = rgb.float() / 255.0  # [3, H, W], float32
    non_white_pixels = ((rgb_float != 1).any(dim=0)).sum().item()
    print(f"count for non white: {non_white_pixels}")

    return rgb_float

def transfer(latent_image):
    img = latent_image.clone()
    top = img[:, :104, :]     # [3, 104, 512]
    bottom = img[:, 104:, :]  # [3, 512, 512]
    # 展平成 [N, 3]
    top_flat = top.permute(1,2,0).reshape(-1, 3)      # [104*512, 3]
    bottom_flat = bottom.permute(1,2,0).reshape(-1, 3) # [512*512, 3]
    # mask
    top_mask = ~(top_flat == 1.0).all(dim=1)          # [N],
    bottom_mask = ~(bottom_flat == 1.0).all(dim=1)    # [N]
    bottom_valid = bottom_flat[bottom_mask]           # [M, 3]

    top_valid_idx = torch.nonzero(top_mask).squeeze(1)   # index list
    top_valid = top_flat[top_mask]

    batch = 4096
    new_color = top_flat.clone()
    for start in range(0, top_valid.shape[0], batch):
        print(start)
        end = min(start + batch, top_valid.shape[0])
        dists = torch.cdist(top_valid[start:end].unsqueeze(0), bottom_valid.unsqueeze(0)).squeeze(0)  # [B, M]
        min_idx = torch.argmin(dists, dim=1)  # [B]
        best_colors = bottom_valid[min_idx]   # [B, 3]
        new_color[top_valid_idx[start:end]] = best_colors

    new_top = new_color.reshape(104, 512, 3).permute(2, 0, 1)  # [3, 104, 512]
    img[:, :104, :] = new_top
    return img


def build_nonwhite_mask(img, threshold=0.98):
    # img: [3, H, W] float [0,1]
    img_np = img.permute(1,2,0).cpu().numpy()  # [H,W,3]
    mask = ~(np.all(img_np >= threshold, axis=-1))  # True=非白
    return mask  # [H,W], bool

from tqdm import tqdm
def match_recon_to_org_by_rgb(recon_img, org_img):
    H, W = recon_img.shape[1:]
    recon_mask = build_nonwhite_mask(recon_img)
    org_mask = build_nonwhite_mask(org_img)
    recon_ys, recon_xs = np.nonzero(recon_mask)
    org_ys, org_xs = np.nonzero(org_mask)
    org_rgbs = org_img[:, org_ys, org_xs].permute(1,0).cpu().numpy()  # [N_org, 3]

    recon_filled = recon_img.clone().cpu().numpy()

    for i in tqdm(range(len(recon_ys)), desc="Matching pixels"):
        y, x = recon_ys[i], recon_xs[i]
        recon_rgb = recon_img[:, y, x].cpu().numpy()  # [3]
        dists = np.linalg.norm(org_rgbs - recon_rgb, axis=1)
        min_idx = np.argmin(dists)
        yy, xx = org_ys[min_idx], org_xs[min_idx]
        recon_filled[:, y, x] = org_img[:, yy, xx].cpu().numpy()

    return recon_filled

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

def analyze_folder_cell_type_colors(folder_path, model, save_dir=None):
    def load_and_recover_z3d_png(path):
        img = Image.open(path).convert("RGB")
        arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
        rgb = arr.view(img.size[1], img.size[0], 3).permute(2, 0, 1).clone()
        mask = (rgb > 250).all(dim=0)
        rgb[:, mask] = 255
        return rgb.float() / 255.0  # [3,H,W]

    def infer_cell_map(latent_image, model):
        min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device)
        range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals
        H, W = latent_image.shape[1:]
        flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3).to("cuda")
        white_mask = (flat_img == 1.0).all(dim=1)
        infer_input_rgb = flat_img[~white_mask]
        pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
        if infer_input_rgb.shape[0] > 0:
            z_recovered = infer_input_rgb * range_vals + min_vals
            logits = model.decoder(z_recovered)
            pred[~white_mask] = torch.argmax(logits, dim=1) + 1
        return pred.reshape(H, W)

    def recolor_image_by_type_centroid(rgb_img, type_map):
        type_map_np = type_map.cpu().numpy()
        img_np = (rgb_img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        recolored_np = np.ones_like(img_np) * 255
        for t in range(1, 26):
            mask = (type_map_np == t)
            if np.any(mask):
                mean_color = img_np[mask].mean(axis=0).astype(np.uint8)
                recolored_np[mask] = mean_color
        return torch.from_numpy(recolored_np).permute(2, 0, 1).float() / 255.0

    type_color_dict = {i: [] for i in range(1, 26)}
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".png")])

    for fname in tqdm(files, desc="Processing folder"):
        path = os.path.join(folder_path, fname)
        rgb_img = load_and_recover_z3d_png(path).to("cuda")
        with torch.no_grad():
            pred_map = infer_cell_map(rgb_img, model).cpu()

        recolored = recolor_image_by_type_centroid(rgb_img, pred_map)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            out_path = os.path.join(save_dir, fname)
            to_save = (recolored.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            Image.fromarray(to_save).save(out_path)

        img_np = (recolored.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        for t in range(1, 26):
            mask = (pred_map.numpy() == t)
            if np.any(mask):
                type_color_dict[t].append(img_np[mask].mean(axis=0))

    print("\n--- Global Mean RGB by Cell Type ---")
    global_mean_colors = {}
    for t in range(1, 26):
        if type_color_dict[t]:
            all_colors = np.stack(type_color_dict[t], axis=0)
            mean_rgb = all_colors.mean(axis=0).round(1)
            global_mean_colors[t] = mean_rgb
            print(f"Type {t}: {mean_rgb.tolist()} (N={len(all_colors)})")
        else:
            global_mean_colors[t] = None
            print(f"Type {t}: None")

    fig_cols = 5
    fig_rows = int(np.ceil(25 / fig_cols))
    fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(fig_cols * 2, fig_rows * 2))
    axes = axes.flatten()
    for t in range(1, 26):
        ax = axes[t - 1]
        if global_mean_colors[t] is not None:
            rgb = global_mean_colors[t] / 255.0
            ax.add_patch(plt.Rectangle((0, 0), 1, 1, color=rgb))
            ax.text(0.5, 0.5, f"Type {t}\n{global_mean_colors[t].astype(int).tolist()}",
                    ha='center', va='center', fontsize=9,
                    color='k' if np.mean(rgb) > 0.7 else 'w')
        else:
            ax.add_patch(plt.Rectangle((0, 0), 1, 1, color='white'))
            ax.text(0.5, 0.5, f"Type {t}\nNone", ha='center', va='center',
                    fontsize=9, color='gray')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
    for i in range(25, len(axes)):
        axes[i].axis('off')

    plt.suptitle("Global Mean RGB Color for Each Cell Type", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    return global_mean_colors

def decode_to_probabilities(latent_image, model):
        """
        latent_image: [3, H, W] RGB image, values in [0, 1]
        model: trained Autoencoder (only decoder is used)

        Returns:
            prob_map: [H, W, 25] tensor, each pixel is a probability vector
                      white pixels are filled with zeros
        """
        min_vals = torch.tensor([-6.290313, -6.6032205, -7.1406803], device="cuda")
        range_vals = torch.tensor([7.1585207, 6.791606, 6.5948086], device="cuda") - min_vals

        H, W = latent_image.shape[1], latent_image.shape[2]
        device = next(model.parameters()).device

        model.eval()
        with torch.no_grad():
            flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3).to(device)

            white_mask = (flat_img == 1.0).all(dim=1)
            infer_input_rgb = flat_img[~white_mask]

            prob_all = torch.zeros(flat_img.shape[0], 25, device=device)

            if infer_input_rgb.shape[0] > 0:
                z_recovered = infer_input_rgb * range_vals + min_vals
                probs = model.decoder(z_recovered)  # [N_nonwhite, 25]
                prob_all[~white_mask] = probs

            prob_map = prob_all.reshape(H, W, 25)

        return prob_map.cpu().numpy()  # shape = [H, W, 25]

In [ ]:
import torch
from scipy.interpolate import make_interp_spline

yticklabels = [
    "1: B",
    "2: CD4+ T cell",
    "3: CD57+ Enterocyte",
    "4: CD66+ Enterocyte",
    "5: CD7+ Immune",
    "6: CD8+ T",
    "7: Cycling TA",
    "8: DC",
    "9: Endothelial",
    "10: Enterocyte",
    "11: Goblet",
    "12: ICC",
    "13: Lymphatic",
    "14: M1 Macrophage",
    "15: M2 Macrophage",
    "16: MUC1+ Enterocyte",
    "17: NK",
    "18: Nerve",
    "19: Neuroendocrine",
    "20: Neutrophil",
    "21: Paneth",
    "22: Plasma",
    "23: Smooth muscle",
    "24: Stroma",
    "25: TA"
]

cell_type_names = {
    1:  "B", 2:  "CD4+ T", 3:  "CD57+ Ent", 4:  "CD66+ Ent", 5:  "CD7+ Imm",
    6:  "CD8+ T", 7:  "Cycling TA", 8:  "DC", 9:  "Endothelial", 10: "Enterocyte",
    11: "Goblet", 12: "ICC", 13: "Lymphatic", 14: "M1 Mac", 15: "M2 Mac",
    16: "MUC1+ Ent", 17: "NK", 18: "Nerve", 19: "Neuroendocrine", 20: "Neutrophil",
    21: "Paneth", 22: "Plasma", 23: "Smooth muscle", 24: "Stroma", 25: "TA"
}
def rgb_centroid_distance_score(map_true, map_gen, type_true, type_gen, visualize=True, penalty_value=2.5):
    """
    Compute centroid distance for each cell type in RGB space.

    Args:
        map_true: torch.Tensor [3, H, W] - RGB map of true image
        map_gen: torch.Tensor [3, H, W] - RGB map of generated image
        type_true: torch.Tensor [1, H, W] - cell type map of true image
        type_gen: torch.Tensor [1, H, W] - cell type map of generated image
        visualize: bool - whether to plot centroids
        penalty_value: float - penalty for missing types

    Returns:
        score: float - average similarity score
    """
    if map_true.shape[0] == 3:
        map_true = map_true.permute(1, 2, 0).cpu().numpy()
    if map_gen.shape[0] == 3:
        map_gen = map_gen.permute(1, 2, 0).cpu().numpy()

    type_true = type_true.squeeze().cpu().numpy()
    type_gen = type_gen.squeeze().cpu().numpy()

    centroids_true = {}
    centroids_gen = {}
    all_types = set(np.unique(type_true)).union(np.unique(type_gen))
    all_types.discard(0)  # ignore background

    for t in all_types:
        coords_true = np.argwhere(type_true == t)
        coords_gen = np.argwhere(type_gen == t)

        if len(coords_true) > 0:
            rgbs_true = map_true[coords_true[:, 0], coords_true[:, 1]]
            centroids_true[t] = rgbs_true.mean(axis=0)

        if len(coords_gen) > 0:
            rgbs_gen = map_gen[coords_gen[:, 0], coords_gen[:, 1]]
            centroids_gen[t] = rgbs_gen.mean(axis=0)

    distances = []
    for t in sorted(all_types):
        if t in centroids_true and t in centroids_gen:
            d = np.linalg.norm(centroids_true[t] - centroids_gen[t])
        else:
            d = penalty_value
        distances.append(d)

    if visualize:
        fig = plt.figure(figsize=(13, 8))
        ax = fig.add_subplot(111, projection='3d')

        for t in sorted(all_types):
            color = centroids_true.get(t, centroids_gen.get(t, [0.5, 0.5, 0.5]))
            if t in centroids_true:
                ax.scatter(*centroids_true[t], marker='o', color=color, s=80, label=cell_type_names[t])
            if t in centroids_gen:
                ax.scatter(*centroids_gen[t], marker='^', color=color, s=80, label=cell_type_names[t])
            if t in centroids_true and t in centroids_gen:
                ax.plot(
                    [centroids_true[t][0], centroids_gen[t][0]],
                    [centroids_true[t][1], centroids_gen[t][1]],
                    [centroids_true[t][2], centroids_gen[t][2]],
                    'k--', alpha=0.4
                )

        ax.set_xlabel('R')
        ax.set_ylabel('G')
        ax.set_zlabel('B')
        ax.set_title("RGB Centroid Distance per Cell Type (3D)")
        ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=9, ncol=2)
        plt.tight_layout()
        plt.show()

    avg_dist = np.mean(distances)
    #print("dist", avg_dist)
    score = kl_to_similarity(avg_dist, min_kl=0.0, max_kl=1)
    return score

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.spatial.distance import jensenshannon
from scipy.optimize import linear_sum_assignment


def catplot2(df, hue, exp='Exp', X='X', Y='Y', invert_y=False, size=3, legend=True, palette="bright", figsize=10,
             style='white', exps=None, axis='on', scatter_kws={}):
    '''
    Plots cells in tissue section color coded by either cell type or node allocation.
    df:  dataframe with cell information
    size:  size of point to plot for each cell.
    hue:  color by "Clusterid" or "Node" respectively.
    legend:  to include legend in plot.
    '''
    scatter_kws_ = {'s': size, 'alpha': 1}
    scatter_kws_.update(scatter_kws)

    figures = []
    df = df.rename(columns=lambda x: str(x))

    df[hue] = df[hue].astype("category")
    if invert_y:
        y_orig = df[Y].values.copy()
        df[Y] *= -1

    style = {'axes.facecolor': style}
    sns.set_style(style)
    if exps == None:
        exps = list(df[exp].unique())  # display all experiments
    elif type(exps) != list:
        exps = [exps]

    for name in exps:
        data = df[df[exp] == name]

        #print(name)
        f = sns.lmplot(x=X, y=Y, data=data, hue=hue,
                       legend=legend, fit_reg=False, markers='.', height=figsize, palette=palette, scatter=True,
                       scatter_kws=scatter_kws_)
        f.ax.set_aspect('equal')
        if axis == 'off':
            sns.despine(top=True, right=True, left=True, bottom=True)
            f = f.set(xticks=[], yticks=[]).set_xlabels('').set_ylabels('')

        plt.title(name)

        plt.show()
        figures += [f]
    if invert_y:
        df[Y] = y_orig

    return figures
from matplotlib.colors import ListedColormap

import matplotlib.pyplot as plt
import numpy as np

def plot_cluster_map(cluster_labels, coords, title, H, W, palette=None, size=3, legend=False):
    coords = np.array(coords)
    x = coords[:, 0]
    y = coords[:, 1]
    labels = cluster_labels

    plt.figure(figsize=(10, 10))
    ax = plt.gca()

    unique_labels = np.unique(labels)

    for label in unique_labels:
        mask = labels == label
        color = palette[label] if palette else None
        ax.scatter(x[mask], y[mask], s=size, c=color, label=f"Cluster {label}", marker='s', linewidth=0)

    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor('white')
    plt.title(title)

    if legend:
        ax.legend(loc='upper right', fontsize=6, frameon=False)

    plt.tight_layout()
    plt.show()


import torch

def compute_ratio(arr, num_types=25):
    if num_types == 25:
        arr = arr.astype(int) - 1  # 转为 0-based
    else:
        arr = arr.astype(int)
    counts = np.bincount(arr, minlength=num_types)
    ratio = counts / counts.sum()
    return ratio
def compute_hist(features, coords, k=20, num_types=25):
    if num_types != 25:
        features = features.astype(int) + 1
    nbrs = NearestNeighbors(n_neighbors=min(k, len(coords))).fit(coords)
    _, indices = nbrs.kneighbors(coords)

    hist = np.zeros((len(coords), num_types), dtype=np.float32)

    for i, neighbors in enumerate(indices):
        for nid in neighbors:
            t = int(features[nid])
            if 1 <= t <= num_types:
                hist[i, t - 1] += 1

    if num_types == 25:
        col_names = [
        "B", "CD4+ T", "CD57+ Ent", "CD66+ Ent", "CD7+ Imm",
        "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
        "Goblet", "ICC", "Lymphatic", "M1 Mac", "M2 Mac",
        "MUC1+ Ent", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
        "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
        ]
        df = pd.DataFrame(hist, columns=col_names)
    else:
        df = pd.DataFrame(hist)

    df["y"] = coords[:, 0]
    df["x"] = coords[:, 1]
    df["cell_type"] = features

    df.to_csv("1.csv", index=False)
    return hist

old_order = [
    "B", "CD4+ T", "CD57+ Ent", "CD66+ Ent", "CD7+ Imm",
    "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
    "Goblet", "ICC", "Lymphatic", "M1 Mac", "M2 Mac",
    "MUC1+ Ent", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
    "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
]

new_order = [
    'Goblet', 'CD66+ Enterocyte', 'Enterocyte', 'Plasma', 'Endothelial',
    'Stroma', 'ICC', 'Smooth muscle', 'Cycling TA', 'M2 Macrophage',
    'Nerve', 'CD4+ T cell', 'Lymphatic', 'TA', 'MUC1+ Enterocyte', 'B',
    'DC', 'CD8+ T', 'NK', 'Neutrophil', 'CD57+ Enterocyte', 'CD7+ Immune',
    'Neuroendocrine', 'M1 Macrophage'
]

name_map = {
    'Goblet': 'Goblet',
    'CD66+ Enterocyte': 'CD66+ Ent',
    'Enterocyte': 'Enterocyte',
    'Plasma': 'Plasma',
    'Endothelial': 'Endothelial',
    'Stroma': 'Stroma',
    'ICC': 'ICC',
    'Smooth muscle': 'Smooth muscle',
    'Cycling TA': 'Cycling TA',
    'M2 Macrophage': 'M2 Mac',
    'Nerve': 'Nerve',
    'CD4+ T cell': 'CD4+ T',
    'Lymphatic': 'Lymphatic',
    'TA': 'TA',
    'MUC1+ Enterocyte': 'MUC1+ Ent',
    'B': 'B',
    'DC': 'DC',
    'CD8+ T': 'CD8+ T',
    'NK': 'NK',
    'Neutrophil': 'Neutrophil',
    'CD57+ Enterocyte': 'CD57+ Ent',
    'CD7+ Immune': 'CD7+ Imm',
    'Neuroendocrine': 'Neuroendocrine',
    'M1 Macrophage': 'M1 Mac',
}

def compute_cluster_celltype_composition(cluster_labels, cell_types, n_clusters=5, n_types=34):
    #print("cluster_labels.shape",cluster_labels.shape)
    #print("cell_types.shape",cell_types.shape)
    #print(np.unique(cluster_labels))
    #print(np.unique(cell_types))
    comp = np.zeros((n_clusters, n_types), dtype=np.float32)
    for i in range(len(cluster_labels)):
        c = cluster_labels[i]
        t = cell_types[i] - 1
        comp[c, t] += 1
    #comp /= (comp.sum(axis=0, keepdims=True) + 1e-8)
    return comp

def raw_proportion_heatmap(comp, title, cell_types_fixed_order, row_order=None):
    if row_order is not None:
        comp = comp[row_order]
        comp = comp / (comp.sum(axis=1, keepdims=True) + 1e-8)
    df = pd.DataFrame(comp, columns=cell_types_fixed_order)
    sns.heatmap(df, vmin=0, vmax=1, cmap='Blues', xticklabels=cell_types_fixed_order, yticklabels=[f"C{i}" for i in range(comp.shape[0])])
    plt.title(title + " (Proportion)")
    plt.xlabel("Cell Type")
    plt.ylabel("Cluster")
    plt.tight_layout()
    plt.show()

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

def compare_cluster_feature_centroids(features1, labels1, features2, labels2, n_clusters, metric='euclidean'):
    dim1 = features1.shape[1]
    dim2 = features2.shape[1]

    if dim1 != dim2:
        print(f"[Warn] feature dim mismatch: features1={dim1}, features2={dim2}")

        max_dim = max(dim1, dim2)

        def pad_to(x, d):
            if x.shape[1] == d:
                return x
            pad_width = d - x.shape[1]
            return np.pad(x, ((0, 0), (0, pad_width)), mode='constant')

        features1 = pad_to(features1, max_dim)
        features2 = pad_to(features2, max_dim)

    global_mean1 = features1.mean(axis=0)
    global_mean2 = features2.mean(axis=0)

    centroids1 = []
    centroids2 = []

    for i in range(n_clusters):
        f1 = features1[labels1 == i]
        f2 = features2[labels2 == i]

        c1 = f1.mean(axis=0) if len(f1) > 0 else global_mean1
        c2 = f2.mean(axis=0) if len(f2) > 0 else global_mean2

        centroids1.append(c1)
        centroids2.append(c2)

    centroids1 = np.stack(centroids1)
    centroids2 = np.stack(centroids2)

    cost_matrix = cdist(centroids1, centroids2, metric=metric)

    if np.any(np.isnan(cost_matrix)) or cost_matrix.shape[0] == 0:
        return [0], [0], np.array([[9999.0]]), 9999.0

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    total_dist = cost_matrix[row_ind, col_ind].sum()
    matched_distances = cost_matrix

    min_dist = matched_distances.min()
    max_dist = matched_distances.max()

    if max_dist > min_dist:
        normalized = (matched_distances - min_dist) / (max_dist - min_dist)
    else:
        normalized = np.zeros_like(matched_distances)

    return row_ind, col_ind, cost_matrix, normalized[row_ind, col_ind].sum()


def log2_fc_heatmap_from_centroids0(niche_clusters, tissue_values, cell_types_fixed_order, k_value, row_order=None):
    tissue_avg = tissue_values#.mean(axis=0, keepdims=True) + 1e-8
    smoothed = niche_clusters + tissue_avg

    smoothed_norm = smoothed / smoothed.sum(axis=1, keepdims=True)
    fc_log = np.log2(smoothed_norm / tissue_avg)
    if row_order is not None:
        fc_log = fc_log[row_order]
    if len(tissue_avg) == 25:
        df = pd.DataFrame(fc_log, columns=cell_types_fixed_order)
    else:
        df = pd.DataFrame(fc_log)
    plt.figure(figsize=(10, 8))
    if len(tissue_avg) == 25:
        sns.clustermap(df, vmin=-3, vmax=3, cmap='bwr',
                xticklabels=cell_types_fixed_order,
                yticklabels=[f"C{i}" for i in range(fc_log.shape[0])])
    else:
        sns.clustermap(df, vmin=-3, vmax=3, cmap='bwr')
    plt.title(f"Log2 FC Heatmap (k={k_value})")
    plt.xlabel("Cell Type")
    plt.ylabel("Cluster Centroid")
    plt.tight_layout()
    plt.show()

def log2_fc_heatmap_from_centroids1(
    niche_clusters, tissue_values, cell_types_fixed_order, k_value, row_order=None
):
    tissue_avg = tissue_values + 1e-8
    print("tissue", tissue_avg)
    print("cluster",niche_clusters)

    smoothed = niche_clusters + tissue_avg

    smoothed_norm = smoothed / (smoothed.sum(axis=1, keepdims=True) + 1e-8)

    fc_log = np.log2(smoothed_norm / tissue_avg)

    if row_order is not None:
        fc_log = fc_log[row_order]

    if len(tissue_avg) == 25:
        df = pd.DataFrame(fc_log, columns=cell_types_fixed_order)
    else:
        df = pd.DataFrame(fc_log)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df.fillna(0, inplace=True)

    if df.isnull().values.any() or np.isinf(df.values).any():
        print("Warning: DataFrame still contains NaN or inf values!")

    plt.figure(figsize=(10, 8))

    if len(tissue_avg) == 25:
        sns.clustermap(
            df,
            vmin=-3,
            vmax=3,
            cmap='bwr',
            xticklabels=cell_types_fixed_order,
            yticklabels=[f"C{i}" for i in range(fc_log.shape[0])]
        )
    else:
        sns.clustermap(
            df,
            vmin=-3,
            vmax=3,
            cmap='bwr'
        )

    plt.title(f"Log2 FC Heatmap (k={k_value})")
    plt.xlabel("Cell Type")
    plt.ylabel("Cluster Centroid")
    plt.tight_layout()
    plt.show()

def log2_fc_heatmap_from_centroids(niche_clusters, tissue_values, cell_types_fixed_order, k_value, row_order=None):
        tissue_avg = tissue_values  # .mean(axis=0, keepdims=True) + 1e-8
        valid_idx = tissue_avg != 0
        tissue_avg = tissue_avg[valid_idx]
        # print(niche_clusters)
        # print(tissue_avg)
        smoothed = niche_clusters + tissue_avg

        smoothed_norm = smoothed / smoothed.sum(axis=1, keepdims=True)
        fc_log = np.log2(smoothed_norm / tissue_avg)
        if row_order is not None:
            fc_log = fc_log[row_order]
        if len(tissue_avg) == 25:
            df = pd.DataFrame(fc_log, columns=cell_types_fixed_order)
        else:
            df = pd.DataFrame(fc_log)
        plt.figure(figsize=(10, 8))
        if len(tissue_avg) == 25:
            sns.heatmap(df, vmin=-3, vmax=3, cmap='bwr',
                        xticklabels=cell_types_fixed_order,
                        yticklabels=[f"C{i}" for i in range(fc_log.shape[0])])
        else:
            sns.heatmap(df, vmin=-3, vmax=3, cmap='bwr')
        plt.title(f"Log2 FC Heatmap (k={k_value})")
        plt.xlabel("Cell Type")
        plt.ylabel("Cluster Centroid")
        plt.tight_layout()
        plt.show()


def print_composition_summary(comp, title, cell_type_names):
    total_proportion = comp.sum(axis=0) / comp.sum()
    print(f"===== {title} =====")
    print("Total Proportion per Cell Type:")
    for i, name in enumerate(cell_type_names):
        print(f"{name}: {total_proportion[i]:.4f}")
    print("\\nPer-Cluster Proportion:")
    df = pd.DataFrame(comp, columns=cell_type_names)
    print(df.round(3))
    print("\\n")

def neighbor_kmeans_composition_matching_score(true_map, gen_map1, gen_map2, gen_map3,
                                               k=50, n_clusters=5, visualize=True):
    maps = [gen_map1, gen_map2, gen_map3]
    scores = []
    ratios = []
    yticklabels = [
        "B", "CD4+ T", "CD57+ Ent", "CD66+ Ent", "CD7+ Imm",
        "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
        "Goblet", "ICC", "Lymphatic", "M1 Mac", "M2 Mac",
        "MUC1+ Ent", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
        "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
    ]

    map_true = true_map.squeeze(0)
    H, W = map_true.shape
    coords_true = np.array([(x, y) for y in range(H) for x in range(W)])
    feats_true = map_true.flatten().cpu().numpy()
    mask_true = feats_true != 0
    feats_true = feats_true[mask_true]
    coords_true = coords_true[mask_true]
    hist_true = compute_hist(feats_true, coords_true, k=k)
    km_true = MiniBatchKMeans(n_clusters=n_clusters, random_state=0).fit(hist_true)
    cluster_labels_true = km_true.labels_
    centers = km_true.cluster_centers_
    #print("centers", centers)
    ratio_true = compute_ratio(feats_true)
    ratios.append(ratio_true)
    #print("ratio_true", ratio_true)
    if visualize:
        plot_cluster_map(cluster_labels_true, coords_true, "True Map Clustered", H, W)
        log2_fc_heatmap_from_centroids1(
            niche_clusters=centers,
            tissue_values=ratio_true,
            cell_types_fixed_order=yticklabels,
            k_value=n_clusters,
            row_order=None
        )

    for i in range(3):
        map_gen = maps[i].squeeze(0)
        coords_gen = np.array([(x, y) for y in range(H) for x in range(W)])
        feats_gen = map_gen.flatten().cpu().numpy()
        mask_gen = feats_gen != 0
        feats_gen = feats_gen[mask_gen]
        coords_gen = coords_gen[mask_gen]
        #print("type_gen1", feats_gen.shape)
        #print("Number of non-zero points:", len(feats_gen))

        hist_gen = compute_hist(feats_gen, coords_gen, k=k)
        ratio_gen = compute_ratio(feats_gen)
        ratios.append(ratio_gen)
        km_gen = MiniBatchKMeans(n_clusters=n_clusters, random_state=0)
        cluster_labels_gen = km_gen.fit_predict(hist_gen)
        centers = km_gen.cluster_centers_
        comp_gen = compute_cluster_celltype_composition(cluster_labels_gen, feats_gen, n_clusters)

        if visualize:
            plot_cluster_map(cluster_labels_gen, coords_gen, f"Gen Map {i + 1} Clustered", H, W)
            df = pd.DataFrame(coords_gen, columns=["X", "Y"])
            df["Clusterid"] = cluster_labels_gen
            df["Exp"] = "1"
            catplot2(df, hue="Clusterid", exp="Exp", X="X", Y="Y", invert_y=True)

        row_order, col_order, cost_matrix, total_dist = compare_cluster_feature_centroids(
            hist_true, cluster_labels_true,
            hist_gen, cluster_labels_gen,
            n_clusters=n_clusters,
            metric='euclidean'
        )

        comp_gen_aligned = comp_gen[col_order]

        if visualize:
            log2_fc_heatmap_from_centroids1(
                niche_clusters=centers,
                tissue_values=ratio_gen,
                cell_types_fixed_order=yticklabels,
                k_value=n_clusters,
                row_order=range(comp_gen_aligned.shape[0])
            )

        scores.append(1/(1+total_dist))

    return scores

def double_neighbor_kmeans_composition_matching_score(true_map, gen_map1, gen_map2, gen_map3,
                                               k=50, n_clusters=5, visualize=True):
    maps = [gen_map1, gen_map2, gen_map3]
    scores = []
    yticklabels = [
        "B", "CD4+ T", "CD57+ Ent", "CD66+ Ent", "CD7+ Imm",
        "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
        "Goblet", "ICC", "Lymphatic", "M1 Mac", "M2 Mac",
        "MUC1+ Ent", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
        "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
    ]

    map_true = true_map.squeeze(0)
    H, W = map_true.shape
    coords_true = np.array([(x, y) for y in range(H) for x in range(W)])
    feats_true = map_true.flatten().cpu().numpy()
    mask_true = feats_true != 0
    feats_true = feats_true[mask_true]
    coords_true = coords_true[mask_true]
    hist_true = compute_hist(feats_true, coords_true, k=20)
    km_true = MiniBatchKMeans(n_clusters=n_clusters, random_state=0).fit(hist_true)
    cluster_labels_true = km_true.labels_
    hist_true = compute_hist(cluster_labels_true, coords_true, k=k, num_types=len(np.unique(cluster_labels_true)))
    km_true = MiniBatchKMeans(n_clusters=n_clusters, random_state=0).fit(hist_true)
    cluster_labels_true1 = km_true.labels_
    centers = km_true.cluster_centers_
    #print(np.unique(cluster_labels_true1))
    ratio_true = compute_ratio(cluster_labels_true1, len(np.unique(cluster_labels_true1)))

    if visualize:
        plot_cluster_map(cluster_labels_true, coords_true, "Lower True Map Clustered", H, W)
        plot_cluster_map(cluster_labels_true1, coords_true, "Higher True Map Clustered", H, W)
        log2_fc_heatmap_from_centroids(
            niche_clusters=centers,
            tissue_values=ratio_true,
            cell_types_fixed_order=yticklabels,
            k_value=n_clusters,
            row_order=None
        )

    for i in range(3):
        print(i)
        map_gen = maps[i].squeeze(0)
        coords_gen = np.array([(x, y) for y in range(H) for x in range(W)])
        feats_gen = map_gen.flatten().cpu().numpy()
        mask_gen = feats_gen != 0
        feats_gen = feats_gen[mask_gen]
        coords_gen = coords_gen[mask_gen]
        #print("type_gen1", feats_gen.shape)
        #print("Number of non-zero points:", len(feats_gen))

        hist_gen = compute_hist(feats_gen, coords_gen, k=20)
        km_gen = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
        cluster_labels_gen = km_gen.fit_predict(hist_gen)
        hist_gen = compute_hist(cluster_labels_gen, coords_gen, k=k, num_types=len(np.unique(cluster_labels_gen)))
        ratio_gen = compute_ratio(cluster_labels_gen, len(np.unique(cluster_labels_gen)))
        km_gen = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
        cluster_labels_gen1 = km_gen.fit_predict(hist_gen)

        centers = km_gen.cluster_centers_
        comp_gen = compute_cluster_celltype_composition(cluster_labels_gen1, cluster_labels_gen, n_clusters, max(np.unique(cluster_labels_gen1)) + 1)

        if visualize:
            plot_cluster_map(cluster_labels_gen, coords_gen, f"Lower Gen Map {i + 1} Clustered", H, W)
            plot_cluster_map(cluster_labels_gen1, coords_gen, f"Higher Gen Map {i + 1} Clustered", H, W)
            df = pd.DataFrame(coords_gen, columns=["X", "Y"])
            df["Clusterid"] = cluster_labels_gen1
            df["Exp"] = "1"
            catplot2(df, hue="Clusterid", exp="Exp", X="X", Y="Y", invert_y=True)
        row_order, col_order, cost_matrix, total_dist = compare_cluster_feature_centroids(
            hist_true, cluster_labels_true1,
            hist_gen, cluster_labels_gen1,
            n_clusters=n_clusters,
            metric='euclidean'
        )

        comp_gen_aligned = comp_gen[col_order]

        if visualize:
            log2_fc_heatmap_from_centroids(
                niche_clusters=centers,
                tissue_values=ratio_gen,
                cell_types_fixed_order=yticklabels,
                k_value=n_clusters,
                row_order=range(comp_gen_aligned.shape[0])
            )

        scores.append(1/(1+total_dist))

    return scores

import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.stats import entropy
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.stats import entropy
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

def cell_density_score_new(true_map, gen_map1, gen_map2, gen_map3,
                           k=20, bin_width=5.0, penalty_value=1.0,
                           alpha=0.5, visualize=True):
    import numpy as np
    from sklearn.neighbors import NearestNeighbors
    from scipy.special import rel_entr  # for KL divergence
    import matplotlib.pyplot as plt

    def get_coords_and_feats(types):
        H, W = types.shape[1:]
        coords = np.array([(i, j) for i in range(H) for j in range(W)])
        feats = types.squeeze(0).flatten().cpu().numpy()
        mask = feats != 0
        return coords[mask], feats[mask]

    def get_distance_distributions(coords, feats):
        dists_by_type = [None] * 25
        for t in range(0, 25):
            points = coords[feats == t]
            if len(points) < k + 1:
                continue
            nbrs = NearestNeighbors(n_neighbors=k + 1).fit(points)
            dists, _ = nbrs.kneighbors(points)
            mean_dists = dists[:, 1:].mean(axis=1)
            dists_by_type[t] = mean_dists
        return dists_by_type

    maps = [true_map, gen_map1, gen_map2, gen_map3]
    all_dists_by_map = [get_distance_distributions(*get_coords_and_feats(m)) for m in maps]

    true_dists_max = [np.max(d) if d is not None else 1.0 for d in all_dists_by_map[0]]
    bin_edges_list = [np.arange(0, max_v + bin_width, bin_width) for max_v in true_dists_max]

    num_gen = 3
    score_matrix = [[] for _ in range(num_gen)]

    for t in range(0, 25):
        all_dists = [d[t] for d in all_dists_by_map]
        if any(x is None for x in all_dists):
            for i in range(num_gen):
                score_matrix[i].append(1 / (1 + penalty_value))
            continue

        bin_edges = bin_edges_list[t]
        hists = []
        for d in all_dists:
            hist, _ = np.histogram(d, bins=bin_edges)
            hist = hist.astype(np.float32) + 1e-8
            hist /= hist.sum()
            hists.append(hist)

        true_hist = hists[0]
        for i in range(num_gen):
            kl = np.sum(rel_entr(true_hist, hists[i + 1]))  # non-normalized KL
            score = 1 / (1 + kl)
            score_matrix[i].append(score)

            if visualize:
                plt.figure(figsize=(6, 3))
                x = bin_edges[:-1]
                plt.plot(x, true_hist, label='True', linewidth=2)
                plt.plot(x, hists[i + 1], label=f'Gen{i+1}', linestyle='--')
                plt.title(f"Cell Type {t} Dist Histogram (True vs Gen{i+1})")
                plt.xlabel("Mean Distance to Neighbors")
                plt.ylabel("Frequency")
                plt.legend()
                plt.tight_layout()
                plt.show()

    type_scores = [np.mean(s) for s in score_matrix]

    def extract_all_distances(cell_map):
        coords = np.argwhere(cell_map.squeeze().cpu().numpy() != 0)
        if len(coords) <= k:
            return None
        nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
        dists, _ = nbrs.kneighbors(coords)
        return dists[:, 1:].mean(axis=1)

    true_dists = extract_all_distances(true_map)
    global_scores = []

    max_global = true_dists.max() if true_dists is not None else 1.0
    bin_edges_global = np.arange(0, max_global + bin_width, bin_width)

    for gen_map in [gen_map1, gen_map2, gen_map3]:
        gen_dists = extract_all_distances(gen_map)
        if true_dists is None or gen_dists is None:
            global_scores.append(1 / (1 + penalty_value))
            continue

        true_hist, _ = np.histogram(true_dists, bins=bin_edges_global)
        gen_hist, _ = np.histogram(gen_dists, bins=bin_edges_global)
        true_hist = true_hist.astype(np.float32) + 1e-8
        gen_hist = gen_hist.astype(np.float32) + 1e-8
        true_hist /= true_hist.sum()
        gen_hist /= gen_hist.sum()

        kl = np.sum(rel_entr(true_hist, gen_hist))
        score = 1 / (1 + kl)
        global_scores.append(score)

    final_scores = [alpha * type_scores[i] + (1 - alpha) * global_scores[i] for i in range(num_gen)]
    return final_scores


from skimage.metrics import structural_similarity as ssim
import lpips

lpips_loss_fn = lpips.LPIPS(net='vgg').to("cuda")

def spatial_structure_score(true_map, gen_map):
    """
    Compute LPIPS perceptual similarity between two RGB maps.
    Input: [3, H, W], float in [0, 1] (normalized image)
    Returns: float score, lower is more similar
    """
    m1 = torch.as_tensor(true_map).squeeze().to("cuda")
    m2 = torch.as_tensor(gen_map).squeeze().to("cuda")

    if m1.ndim != 3 or m1.shape[0] != 3:
        raise ValueError("Input map must have shape [3, H, W]")

    # Normalize from [0,1] to [-1,1] as required by LPIPS
    m1 = m1 * 2 - 1
    m2 = m2 * 2 - 1

    m1 = m1.unsqueeze(0)  # Add batch dimension: [1, 3, H, W]
    m2 = m2.unsqueeze(0)

    with torch.no_grad():
        score = lpips_loss_fn(m1, m2).item()

    return 1 - score


def kl_to_similarity(kl_div, min_kl=0.0, max_kl=2.0):

    sim = 1.0 - (kl_div - min_kl) / (max_kl - min_kl)

    return np.clip(sim, 0.0, 1.0)



def cell_type_distribution(true_map, generated_map, show_plot=True, type_color_dict=None):
    type_color_dict = {
        1: [134.9, 124.1, 8.7],
        2: [62.3, 216.0, 132.0],
        3: [130.9, 46.4, 182.5],
        4: [92.8, 63.6, 202.5],
        5: [140.1, 239.7, 189.0],
        6: [200.0, 183.1, 168.9],
        7: [108.3, 16.1, 142.3],
        8: [132.7, 217.1, 223.4],
        9: [170.6, 96.0, 98.0],
        10: [68.4, 101.1, 101.3],
        11: [4.6, 146.9, 145.6],
        12: [177.1, 1.5, 112.2],
        13: [202.6, 136.4, 12.3],
        14: [95.1, 244.3, 209.6],
        15: [32.5, 214.2, 217.8],
        16: [176.2, 78.6, 193.2],
        17: [185.9, 206.0, 77.8],
        18: [212.0, 39.3, 35.0],
        19: [117.1, 190.4, 53.4],
        20: [101.2, 145.2, 246.9],
        21: [13.7, 179.1, 133.2],
        22: [247.4, 125.5, 111.5],
        23: [122.9, 151.0, 150.0],
        24: [118.0, 45.6, 47.7],
        25: [31.1, 114.7, 222.5]
    }
    """
    Compare the global cell type distribution between the true and generated maps,
    using KL divergence. Each bar is colored by cell type RGB; true uses solid bars, generated uses hatched.

    Args:
        true_map: torch.Tensor, shape [1, H, W]
        generated_map: torch.Tensor, shape [1, H, W]
        show_plot: bool
        type_color_dict: dict[int -> list of RGB in 0-255], key=cell type (1~25)

    Returns:
        similarity_score: float
    """
    import numpy as np
    import matplotlib.pyplot as plt

    cell_type_names = [
        "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
        "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte", "Goblet", "ICC",
        "Lymphatic", "M1 Macrophage", "M2 Macrophage", "MUC1+ Enterocyte", "NK",
        "Nerve", "Neuroendocrine", "Neutrophil", "Paneth", "Plasma",
        "Smooth muscle", "Stroma", "TA"
    ]

    true_map = true_map.squeeze().cpu().numpy()
    generated_map = generated_map.squeeze().cpu().numpy()

    unique_true = np.unique(true_map)
    unique_gen = np.unique(generated_map)
    all_labels = np.union1d(unique_true, unique_gen)
    all_labels = all_labels[all_labels != 0]

    if all_labels.size == 0:
        return 1.0

    num_classes = int(all_labels.max()) + 1
    true_hist, _ = np.histogram(true_map, bins=np.arange(1, num_classes + 1))
    gen_hist, _ = np.histogram(generated_map, bins=np.arange(1, num_classes + 1))

    if show_plot:
        # Normalize histograms to ratio
        true_ratio = true_hist / (true_hist.sum() + 1e-8)
        gen_ratio = gen_hist / (gen_hist.sum() + 1e-8)

        x = np.arange(1, num_classes)
        fig, ax = plt.subplots(figsize=(12, 4))

        default_color = (0.6, 0.6, 0.6)
        xticks = []
        for i in x:
            # bar color
            color = (np.array(type_color_dict[i]) / 255.0) if (
                        type_color_dict and i in type_color_dict) else default_color
            label = cell_type_names[i - 1] if 1 <= i <= len(cell_type_names) else f"Type {i}"

            # solid bar for true
            ax.bar(i - 0.18, true_ratio[i - 1], width=0.35, label=f'True-{label}' if i == 1 else "",
                   color=color, edgecolor='black')

            # hatched bar for generated
            ax.bar(i + 0.18, gen_ratio[i - 1], width=0.35, label=f'Generated-{label}' if i == 1 else "",
                   color=color, edgecolor='black', hatch='///', linewidth=0.8)

            xticks.append(label)

        ax.set_xticks(x)
        ax.set_xticklabels(xticks, rotation=45, ha='right')
        ax.set_xlabel("Cell Type")
        ax.set_ylabel("Proportion")
        ax.set_ylim(0, 1.05 * max(true_ratio.max(), gen_ratio.max()))
        ax.set_title("Cell Type Distribution (Ratio): True (solid) vs Generated (hatched)")
        from matplotlib.patches import Patch
        ax.legend(handles=[
            Patch(facecolor='gray', edgecolor='black', label='True'),
            Patch(facecolor='gray', edgecolor='black', hatch='///', label='Generated')
        ], loc='upper left')
        plt.tight_layout()
        plt.show()

    if true_hist.sum() == 0 or gen_hist.sum() == 0:
        return 0.0

    eps = 1e-8
    true_hist = (true_hist + eps) / (true_hist.sum() + eps * len(true_hist))
    gen_hist = (gen_hist + eps) / (gen_hist.sum() + eps * len(gen_hist))

    missing_mask = (true_hist > eps) & (gen_hist <= eps)
    kl_div = np.sum(true_hist * np.log(true_hist / (gen_hist + eps)))
    penalty_weight = 2

    missing_penalty = penalty_weight * np.sum(true_hist[missing_mask])
    #print(missing_penalty)
    kl_div += missing_penalty
    #print("KL divergence =", kl_div)

    similarity_score = 1 / (1 + kl_div*kl_div)
    return similarity_score

In [ ]:

# eval_pipeline.py
import torch
from evaluation import Autoencoder, Autoencoder_merfish, load_and_recover_z3d_png, infer_cell_map
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

cell_type_names = [
    "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
    "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte", "Goblet", "ICC",
    "Lymphatic", "M1 Macrophage", "M2 Macrophage", "MUC1+ Enterocyte", "NK",
    "Nerve", "Neuroendocrine", "Neutrophil", "Paneth", "Plasma",
    "Smooth muscle", "Stroma", "TA"
]

def save_inferred_map_to_df(cell_type_map: torch.Tensor, region_id, save_path=None) -> pd.DataFrame:
    if cell_type_map.ndim == 3:
        cell_type_map = cell_type_map.squeeze(0)  # [H, W]

    cell_type_np = cell_type_map.cpu().numpy().astype(int)  # [H, W]
    H, W = cell_type_np.shape

    ys, xs = np.where(cell_type_np > 0)
    types = cell_type_np[ys, xs]

    type_names = [cell_type_names[t - 1] if 1 <= t <= len(cell_type_names) else "Unknown" for t in types]

    df = pd.DataFrame({
        "Cell Type": type_names,
        "x": xs,
        "y": ys,
        "unique_region": region_id
    })

    if save_path is not None:
        df.to_csv(save_path, index=False)
        print(f"Saved inferred map CSV to: {save_path}")

    return df
def print_cell_type_centroid_colors(img, type_map, img_name="Map"):
    """
    img: torch.Tensor or np.ndarray, shape [3,H,W] or [H,W,3], 0-1/0-255
    type_map: torch.Tensor or np.ndarray, shape [H,W]
    """
    if hasattr(img, 'cpu'):
        img = img.cpu().numpy()
    if hasattr(type_map, 'cpu'):
        type_map = type_map.cpu().numpy()
    if img.shape[0] == 3:
        img = np.transpose(img, (1, 2, 0))
    if img.max() <= 1.1:
        img = (img * 255).round().astype(np.uint8)

    H, W, _ = img.shape
    cell_types = sorted(set(np.unique(type_map)) - {0})

    print(f"--- {img_name} ---")
    fig, axes = plt.subplots(len(cell_types), 1, figsize=(6, 1.2 * len(cell_types)))
    if len(cell_types) == 1:
        axes = [axes]

    for i, t in enumerate(cell_types):
        mask = (type_map == t)
        mask = np.squeeze(mask)
        if not np.any(mask):
            continue
        colors = img[mask]  # [N, 3]
        center = colors.mean(axis=0)
        print(f"Cell type {t}: mean RGB={center.round(1).tolist()} (N={len(colors)})")
        ax = axes[i]
        rgb = center / 255  # matplotlib expects 0-1
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, color=rgb))
        txt = f"RGB={center.round(1).astype(int).tolist()}\nN={len(colors)}"
        ax.text(0.5, 0.5, txt, ha='center', va='center', fontsize=11,
                color='k' if np.mean(rgb) > 0.7 else 'w')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        ax.set_title(f"Cell type {t}", fontsize=12)
    plt.suptitle(f"{img_name} cell type mean colors")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


import torch
import numpy as np
import torch
from PIL import Image

def assign_type_by_nearest_color_centroid(true_img: torch.Tensor, true_type_map: torch.Tensor, target_img_path: str, threshold=0.68):
    # Step 1:
    img_np = (true_img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)  # [H, W, 3]
    type_map_np = true_type_map.squeeze(0).cpu().numpy()  # [H, W]

    cell_types = sorted(set(np.unique(type_map_np)) - {0})
    centroids = []

    for t in cell_types:
        mask = (type_map_np == t)
        pixels = img_np[mask]
        if len(pixels) == 0:
            centroids.append(np.array([255, 255, 255]))  # 占位
        else:
            center = pixels.mean(axis=0)
            centroids.append(center)
    centroids = np.stack(centroids, axis=0)  # [N_types, 3]

    # Step 2:
    img = Image.open(target_img_path).convert("RGB")
    target_rgb = np.array(img)  # [H, W, 3]
    h, w, _ = target_rgb.shape

    target_rgb_norm = target_rgb.astype(np.float32)  # [H, W, 3]
    flat_rgb = target_rgb_norm.reshape(-1, 3)  # [H*W, 3]

    white_mask = (flat_rgb >= threshold * 255).all(axis=1)  # [H*W]
    type_pred_flat = np.zeros(h * w, dtype=np.uint8)

    nonwhite_idx = np.where(~white_mask)[0]
    nonwhite_rgb = flat_rgb[~white_mask]  # [N, 3]

    dists = np.linalg.norm(nonwhite_rgb[:, None, :] - centroids[None, :, :], axis=2)  # [N, N_types]
    min_indices = np.argmin(dists, axis=1)  # [N]
    mapped_types = np.array(cell_types)[min_indices]  # [N]

    type_pred_flat[nonwhite_idx] = mapped_types
    type_pred_map = type_pred_flat.reshape(1, h, w)

    return torch.tensor(type_pred_map, dtype=torch.long)

def evaluate_generated_maps(true_img_path, gen_img_path1, gen_img_path2, gen_img_path3, model_path, i):
    model = Autoencoder().to("cuda")
    model.load_state_dict(torch.load(model_path))
    model.eval()

    true_map = load_and_recover_z3d_png(true_img_path).to("cuda")
    gen_map1 = load_and_recover_z3d_png(gen_img_path1).to("cuda")
    gen_map2 = load_and_recover_z3d_png(gen_img_path2).to("cuda")
    gen_map3 = load_and_recover_z3d_png(gen_img_path3).to("cuda")

    type_true = infer_cell_map(true_map, model)
    df_true = save_inferred_map_to_df(type_true, region_id=i)
    type_gen1 = infer_cell_map(gen_map1, model)
    df_gen1 = save_inferred_map_to_df(type_gen1, region_id=i)
    type_gen2 = infer_cell_map(gen_map2, model)
    df_gen2 = save_inferred_map_to_df(type_gen2, region_id=i)
    type_gen3 = infer_cell_map(gen_map3, model)
    df_gen3 = save_inferred_map_to_df(type_gen3, region_id=i)


    s1_1 = rgb_centroid_distance_score(true_map, gen_map1, type_true, type_gen1, visualize=False)
    s1_2 = rgb_centroid_distance_score(true_map, gen_map2, type_true, type_gen2, visualize=False)
    s1_3 = rgb_centroid_distance_score(true_map, gen_map3, type_true, type_gen3, visualize=False)
    s2 = neighbor_kmeans_composition_matching_score(type_true, type_gen1, type_gen2, type_gen3, k=10, n_clusters=20, visualize=False)
    s3 = cell_density_score_new(type_true, type_gen1, type_gen2, type_gen3, k=50, visualize=False)

    s4_1 = spatial_structure_score(true_map, gen_map1)
    s4_2 = spatial_structure_score(true_map, gen_map2)
    s4_3 = spatial_structure_score(true_map, gen_map3)

    s5_1 = cell_type_distribution(type_true, type_gen1, show_plot=False)
    s5_2 = cell_type_distribution(type_true, type_gen2, show_plot=False)
    s5_3 = cell_type_distribution(type_true, type_gen3, show_plot=False)

    return [s1_1, s1_2, s1_3], s2, s3, [s4_1, s4_2, s4_3], [s5_1, s5_2, s5_3]


if __name__ == '__main__':
    results = []

    with (open("evaluation_results_new.txt", "w") as f):
        for i in range(1,32):

            umap_scores, kmeans_scores, density_scores, ss_scores, dist_scores = evaluate_generated_maps(
            f"C:/Users/18983/Desktop/pixelcut/{i}_bottom.png", #change this
            f"C:/Users/18983/Desktop/pixelcut/{i}_top.png",
            f"C:/Users/18983/Desktop/pixelcut/{i}a.png",
            f"C:/Users/18983/Desktop/pixelcut/{i}b.png",
            "newae2.pth"
            )

            f.write(f"Feature Score:{umap_scores}\n")
            f.write(f"Kmeans Score:{kmeans_scores}\n")
            f.write(f"Cell Density Score: {density_scores}\n")
            f.write(f"SS Score: {ss_scores}\n")
            f.write(f"Cell Type Distribution Score: {dist_scores}\n\n")

            results.append([*umap_scores, *kmeans_scores, *density_scores, *ss_scores, *dist_scores])

        results_tensor = torch.tensor(results)
        avg = results_tensor.mean(dim=0).tolist()

        f.write("=== AVERAGE ===\n")
        f.write(f"UMAP Cluster Score: {avg[0]:.4f}, {avg[1]:.4f}, {avg[2]:.4f}\n")
        f.write(f"Neighborhood KMeans Score: {avg[3]:.4f}, {avg[4]:.4f}, {avg[5]:.4f}\n")
        f.write(f"Cell Density Score: {avg[6]:.4f}, {avg[7]:.4f}, {avg[8]:.4f}\n")
        f.write(f"SS Score: {avg[9]:.4f}, {avg[10]:.4f}, {avg[11]:.4f}\n")
        f.write(f"Cell Type Distribution Score: {avg[12]:.4f}, {avg[13]:.4f}, {avg[14]:.4f}\n")

